# Track A Fine-Tuning on Kaggle (Deformable DETR)

This notebook is designed for Kaggle GPU notebooks and the COCO dataset:
- Dataset: `awsaf49/coco-2017-dataset`
- Goal: fine-tune Track A (variance head) for uncertainty estimation


In [4]:
import os, sys, subprocess, json, shutil
print('Python:', sys.version)
print('Working dir:', os.getcwd())
os.environ['TOKENIZERS_PARALLELISM'] = 'false'


Python: 3.12.12 (main, Oct 10 2025, 08:52:57) [GCC 11.4.0]
Working dir: /kaggle/working


In [5]:
!nvidia-smi || true
print('GPU check done. Torch import will be verified after dependency install.')


/bin/bash: line 1: nvidia-smi: command not found
GPU check done. Torch import will be verified after dependency install.


## 1) Get code

In [6]:
REPO_URL = 'https://github.com/krishna-kumar-bais/CS776_Project.git'
REPO_DIR = '/kaggle/working/CV_Project'

if os.path.exists(REPO_DIR):
    shutil.rmtree(REPO_DIR)

!git clone "$REPO_URL" "$REPO_DIR"
print('Cloned:', REPO_DIR)


Cloning into '/kaggle/working/CV_Project'...
remote: Enumerating objects: 87, done.
remote: Counting objects: 100% (87/87), done.
remote: Compressing objects: 100% (85/85), done.
remote: Total 87 (delta 21), reused 4 (delta 0), pack-reused 0 (from 0)
Receiving objects: 100% (87/87), 16.83 MiB | 25.31 MiB/s, done.
Resolving deltas: 100% (21/21), done.
Cloned: /kaggle/working/CV_Project


In [7]:
%cd /kaggle/working/CV_Project/Deformable-DETR
!pwd
!ls


/kaggle/working/CV_Project/Deformable-DETR
/kaggle/working/CV_Project/Deformable-DETR
benchmark.py  datasets	engine.py  LICENSE  models     requirements.txt  util
configs       docs	figs	   main.py  README.md  tools


## 2) Install dependencies


In [8]:
!python -m pip install --upgrade pip
!python -m pip install -r requirements.txt
!python -m pip install packaging


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 21.8 MB/s eta 0:00:0000:010:01
  Attempting uninstall: pip
    Found existing installation: pip 24.1.2
    Uninstalling pip-24.1.2:
      Successfully uninstalled pip-24.1.2


## 3) Use attached COCO dataset from Kaggle Input

Dataset: https://www.kaggle.com/datasets/awsaf49/coco-2017-dataset
This notebook expects the dataset to be attached in the notebook's **Input** panel.


In [9]:
import os

dataset_candidates = [
    '/kaggle/input/coco-2017-dataset/coco2017',
    '/kaggle/input/datasets/awsaf49/coco-2017-dataset/coco2017',
    '/kaggle/input/datasets/awsaf49/coco-2017-dataset',
]

dataset_path = next((p for p in dataset_candidates if os.path.exists(p)), None)
if dataset_path is None:
    raise FileNotFoundError(
        'COCO dataset not found. Attach awsaf49/coco-2017-dataset in Kaggle Input first.'
    )

os.makedirs('/kaggle/working/CV_Project/Deformable-DETR/data', exist_ok=True)
coco_link = '/kaggle/working/CV_Project/Deformable-DETR/data/coco'
if os.path.islink(coco_link) or os.path.exists(coco_link):
    !rm -rf "$coco_link"
!ln -s "$dataset_path" "$coco_link"

print('Using COCO dataset path:', dataset_path)
print('Linked COCO ->', coco_link)
!ls -la /kaggle/working/CV_Project/Deformable-DETR/data/coco


Using COCO dataset path: /kaggle/input/datasets/awsaf49/coco-2017-dataset/coco2017
Linked COCO -> /kaggle/working/CV_Project/Deformable-DETR/data/coco
lrwxrwxrwx 1 root root 57 Apr 18 08:33 /kaggle/working/CV_Project/Deformable-DETR/data/coco -> /kaggle/input/datasets/awsaf49/coco-2017-dataset/coco2017


## 4) Build CUDA operators (required for fast training)


In [10]:
import subprocess

# Set True to never run make.sh (saves time on P100 / broken builds).
FORCE_PYTORCH_FALLBACK = True

def _try_import_msda():
    try:
        import MultiScaleDeformableAttention  # noqa: F401
        return True
    except ImportError:
        return False

def _gpu_major_capability():
    import torch
    if not torch.cuda.is_available():
        return None
    return torch.cuda.get_device_capability(0)[0]

if _try_import_msda():
    print('MultiScaleDeformableAttention already available — skipping build.')
elif FORCE_PYTORCH_FALLBACK:
    print('FORCE_PYTORCH_FALLBACK=True — skipping CUDA ops build. Using pure PyTorch deformable attention (slower).')
else:
    major = _gpu_major_capability()
    if major is not None and major < 7:
        print(
            'GPU compute capability sm_%d0 is below PyTorch’s supported minimum (sm_70) on this image. '
            'Skipping CUDA ops build. Use Kaggle accelerator T4 (or better) if you want native CUDA ops + full GPU speed.'
            % major
        )
    else:
        import os
        os.chdir('/kaggle/working/CV_Project/Deformable-DETR/models/ops')
        ret = subprocess.run('sh make.sh', shell=True)
        if ret.returncode == 0:
            print('CUDA ops build succeeded.')
            !python test.py
        else:
            print('CUDA ops build failed. Continuing with PyTorch fallback (slower but works).')


FORCE_PYTORCH_FALLBACK=True — skipping CUDA ops build. Using pure PyTorch deformable attention (slower).


## 5) Confirm logs/checkpoints are created

In [11]:
import os
import urllib.request
from pathlib import Path

CKPT_PATH = '/kaggle/input/models/gate2024/checkpoint0000-pth/pytorch/default/1/checkpoint.pth'
os.makedirs(os.path.dirname(CKPT_PATH), exist_ok=True)

# Check if already exists
if os.path.exists(CKPT_PATH):
    size_gb = os.path.getsize(CKPT_PATH) / 1e9
    print(f'✓ Checkpoint already exists: {CKPT_PATH}')
    print(f'  Size: {size_gb:.2f} GB')
else:
    print('Downloading checkpoint...')
    
    # Direct download from CivArchive (reliable mirror)
    checkpoint_url = 'https://civarchive.com/files/r50_deformable_detr-checkpoint.pth'
    
    try:
        urllib.request.urlretrieve(checkpoint_url, CKPT_PATH)
        size_gb = os.path.getsize(CKPT_PATH) / 1e9
        print(f'✓ Checkpoint downloaded successfully!')
        print(f'  Location: {CKPT_PATH}')
        print(f'  Size: {size_gb:.2f} GB')
    except Exception as e:
        print(f'✗ Download failed: {e}')
        print('\nAlternative: Manually download and upload to Kaggle')
        print('  1. Download: https://civarchive.com/files/r50_deformable_detr-checkpoint.pth')
        print('  2. Upload as Kaggle Dataset')
        print('  3. Attach to this notebook')

✓ Checkpoint already exists: /kaggle/input/models/gate2024/checkpoint0000-pth/pytorch/default/1/checkpoint.pth
  Size: 0.16 GB


In [12]:
import shutil
import os

SRC_CKPT = '/kaggle/input/models/krishnakb24/r50-deformable-detr-checkpoint-pth/pytorch/default/1/r50_deformable_detr-checkpoint.pth'
DEST_CKPT = '/kaggle/working/CV_Project/Deformable-DETR/models/checkpoint0000.pth'

# shutil.copy(SRC_CKPT, DEST_CKPT)
# Create directory if needed
os.makedirs(os.path.dirname(DEST_CKPT), exist_ok=True)

# Copy checkpoint
print(f'Copying checkpoint...')
print(f'From: {SRC_CKPT}')
print(f'To:   {DEST_CKPT}')

shutil.copy(SRC_CKPT, DEST_CKPT)

# Verify
size_gb = os.path.getsize(DEST_CKPT) / 1e9
print(f'\n✓ Checkpoint copied successfully!')
print(f'  Size: {size_gb:.2f} GB')

Copying checkpoint...
From: /kaggle/input/models/krishnakb24/r50-deformable-detr-checkpoint-pth/pytorch/default/1/r50_deformable_detr-checkpoint.pth
To:   /kaggle/working/CV_Project/Deformable-DETR/models/checkpoint0000.pth

✓ Checkpoint copied successfully!
  Size: 0.48 GB


## 6) Track A fine-tuning

In [ ]:
%cd /kaggle/working/CV_Project/Deformable-DETR

!python main.py \
  --dataset_file coco \
  --coco_path ./data/coco \
  --resume ./models/r50_deformable_detr-checkpoint.pth \
  --output_dir "/kaggle/working/track_a_run1" \
  --device cuda \
  --batch_size 2 \
  --epochs 10 \
  --num_workers 2 \
  --lr 2e-6 \
  --track_a \
  --track_a_loss_coef 0.01 \
  --log_var_min -4.0 \
  --log_var_max 4.0 \
  --clip_max_norm 0.01